## 📦 Oppsett og avhengigheter

Følg stegene under for å sette opp miljøet. Kommandoene kan kopieres fra Python-cellene nedenfor.

### 1. Hent `.env` fil med API-nøkler

> ⚠️ **Viktig:** API-nøklene er kun for workshop-bruk!

1. Gå til: **https://yopass.hafslundeco.no/#/s/cc4582ca-c615-4c6d-a1e5-1a75e14fbd17**
2. Skriv inn dekrypteringsnøkkel: `1234`
3. Kopier innholdet og lagre som `.env` i prosjektmappen

### 2. Opprett og aktiver virtuelt miljø

Åpne en terminal i prosjektmappen og kjør kommandoene fra cellene under.

In [1]:
# Opprett venv (kjør i terminal)
# python3 -m venv .venv

In [2]:
# Aktiver miljøet (kjør i terminal)

# macOS/Linux:
# source .venv/bin/activate

# Windows PowerShell:
# .\.venv\Scripts\Activate.ps1

# Windows CMD:
# .\.venv\Scripts\activate.bat

In [3]:
# Installer pakker (kjør i terminal)
# pip install openai chromadb tiktoken python-dotenv httpx fastmcp rich pymupdf langchain-text-splitters langchain-mcp-adapters langchain-openai langgraph

### 3. Velg kernel i VS Code

1. Klikk på kernel-velgeren øverst til høyre i notebooken
2. Velg **"Select Another Kernel..."** → **"Python Environments..."**
3. Velg **.venv**

---

In [4]:
import os
from dotenv import load_dotenv

# Last inn miljøvariabler fra .env fil
load_dotenv()

# Sjekk at nødvendige Azure-variabler er satt
required_vars = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_MODEL",
    "AZURE_OPENAI_EMBED_MODEL"
]

missing = [var for var in required_vars if not os.getenv(var)]

if missing:
    print(f"⚠️  Mangler følgende miljøvariabler i .env:")
    for var in missing:
        print(f"   - {var}")
else:
    print("✅ Alle Azure OpenAI-variabler funnet!")

✅ Alle Azure OpenAI-variabler funnet!


---

# RAG - Retrieval Augmented Generation

## Hva er RAG?

**RAG** (Retrieval Augmented Generation) er en teknikk som lar språkmodeller svare på spørsmål basert på **dine egne data**, uten å måtte trene modellen på nytt.

### Hvorfor RAG?

Språkmodeller har noen begrensninger:
- 🕐 **Utdatert kunnskap** - Modellen vet kun det den ble trent på
- 🏢 **Mangler bedriftsspesifikk info** - Kjenner ikke dine interne dokumenter
- 🎭 **Hallusinasjoner** - Kan finne på svar når den ikke vet

RAG løser dette ved å **hente relevant kontekst** før modellen genererer svar.

### Hvordan fungerer RAG?

```
┌─────────────────────────────────────────────────────────────────┐
│                        RAG Pipeline                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. INDEKSERING                                                 │
│     ┌──────────┐    ┌──────────────┐    ┌──────────────┐        │
│     │Dokumenter│ -> │ Chunking     │ -> │ Embeddings   │        │
│     │(PDF,TXT) │    │ (del opp)    │    │ (vektorer)   │        │
│     └──────────┘    └──────────────┘    └──────┬───────┘        │
│                                                 │               │
│                                                 v               │ 
│                                         ┌──────────────┐        │
│                                         │ VektorDB     │        │
│                                         │ (ChromaDB)   │        │
│                                         └──────┬───────┘        │
│                                                │                │
│  2. SPØRRING                                   │                │
│     ┌──────────┐    ┌──────────────┐           │                │
│     │ Bruker   │ -> │ Embed query  │ ──────────┘                │
│     │ spørsmål │    └──────────────┘      (søk)                 │
│     └──────────┘                            │                   │
│                                             v                   │
│                     ┌──────────────────────────────────┐        │
│                     │ Relevante chunker returneres     │        │
│                     └──────────────┬───────────────────┘        │
│                                    │                            │
│  3. GENERERING                     v                            │
│     ┌────────────────────────────────────────────────┐          │
│     │  Prompt = Spørsmål + Kontekst fra dokumenter   │          │
│     └────────────────────────┬───────────────────────┘          │
│                              v                                  │
│                       ┌──────────────┐                          │
│                       │   LLM        │                          │
│                       │              │                          │
│                       └──────┬───────┘                          │
│                              v                                  │
│                       ┌──────────────┐                          │
│                       │    Svar      │                          │
│                       └──────────────┘                          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 🛠️ Bygge en enkel RAG!

### Steg 1: Sett opp klienter

In [5]:
from openai import AzureOpenAI
import chromadb

# Sett opp Azure OpenAI klient
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

# Deployment-navn for modellene
CHAT_MODEL = os.getenv("AZURE_OPENAI_CHAT_MODEL")
EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBED_MODEL")

# Sett opp ChromaDB (vektordatabase)
chroma_client = chromadb.Client()

# Lag (eller hent eksisterende) collection for våre dokumenter
# Slett først hvis den finnes, så vi starter med en tom collection
try:
    chroma_client.delete_collection(name="workshop_docs")
except:
    pass

collection = chroma_client.create_collection(
    name="workshop_docs",
    metadata={"hnsw:space": "cosine"}  # Bruk cosine similarity
)

print("✅ Azure OpenAI og ChromaDB klienter satt opp!")
print(f"   📝 Chat model: {CHAT_MODEL}")
print(f"   🔢 Embedding model: {EMBEDDING_MODEL}")

✅ Azure OpenAI og ChromaDB klienter satt opp!
   📝 Chat model: gpt-5.2-chat
   🔢 Embedding model: text-embedding-3-large


### Steg 2: Last inn og chunk PDF-dokumentet

For å jobbe med PDF-er må vi:
1. **Lese** PDF-filen og ekstrahere tekst
2. **Chunke** teksten - dele den opp i mindre biter som passer i LLM-konteksten
3. **Indeksere** hver chunk med embeddings

La oss først installere en PDF-leser:

In [6]:
import fitz  # PyMuPDF
from pathlib import Path

# Last inn PDF
pdf_path = Path("data/Hafslund Delårsrapport per tredje kvartal 2025.pdf")

# Ekstraher tekst fra alle sider
doc = fitz.open(pdf_path)
full_text = ""
page_count = len(doc)  # Lagre antall sider FØR vi lukker dokumentet

for page_num, page in enumerate(doc):
    full_text += f"\n--- Side {page_num + 1} ---\n"
    full_text += page.get_text()

doc.close()

print(f"📄 Lastet: {pdf_path.name}")
print(f"📊 Antall sider: {page_count}")
print(f"📝 Total tekstlengde: {len(full_text):,} tegn")
print(f"\n🔍 Forhåndsvisning av starten:")
print("-" * 50)
print(full_text[:1000])

📄 Lastet: Hafslund Delårsrapport per tredje kvartal 2025.pdf
📊 Antall sider: 19
📝 Total tekstlengde: 40,336 tegn

🔍 Forhåndsvisning av starten:
--------------------------------------------------

--- Side 1 ---
Delårsrapport          
per tredje kvartal 2025

--- Side 2 ---
Introduksjon
Hafslund er et fornybarkonsern bestående av tre forretningsområder: 
Kraftproduksjon, med Norges nest største vannkraftvirksomhet, 
Fjernvarme, som er Norges største fjernvarmeleverandør og Vekst og 
investeringer, som jobber med etablerte og nye vekstinitiativer innenfor 
den fornybare verdikjeden, i tillegg til å omfatte eierskapet i Eidsiva Energi 
med en eierandel på 50 prosent. Eidsiva Energi eier 100 prosent av Elvia, 
Norges største nettselskap.
Gjennomgående i denne rapporten vises sammenligningstall fra i fjor i 
parentes, med mindre annet er oppgitt. 
Resultat og resultatdrivere per tredje kvartal 2025 
• Hafslund fikk et resultat etter skatt per tredje kvartal 2025 på 2 829 
millioner kroner 

### Steg 3: Chunking med LangChain

**Chunking** er prosessen med å dele opp et dokument i mindre biter. Vi bruker LangChain sin `RecursiveCharacterTextSplitter` som er "best practice":

- Prøver å splitte ved naturlige grenser (`\n\n`, `\n`, `. `, etc.)
- Støtter overlapping mellom chunks
- Returnerer `Document`-objekter med tekst og metadata

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Opprett text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      # Maks tegn per chunk
    chunk_overlap=150    # Overlapping mellom chunks
)

# Split teksten - returnerer liste med Document-objekter
docs = text_splitter.create_documents(
    texts=[full_text],
    metadatas=[{"source": pdf_path.name}]
)

# Konverter til vårt format for enkel bruk videre
chunks = [
    {
        "id": f"chunk_{i}",
        "text": doc.page_content,
        "metadata": {**doc.metadata, "chunk_index": i}
    }
    for i, doc in enumerate(docs)
]

print(f"📦 Antall chunks: {len(chunks)}")
print(f"📏 Gjennomsnittlig chunk-størrelse: {sum(len(c['text']) for c in chunks) // len(chunks)} tegn")

📦 Antall chunks: 69
📏 Gjennomsnittlig chunk-størrelse: 678 tegn


In [8]:
# 🔍 Inspiser chunks - se tekst og tilhørende embedding!

def get_embedding(text: str) -> list[float]:
    """Hent embedding for en tekst via Azure OpenAI."""
    response = client.embeddings.create(
        input=text,
        model=EMBEDDING_MODEL
    )
    return response.data[0].embedding

print("=" * 70)
print("📄 CHUNK + EMBEDDING INSPEKSJON")
print("=" * 70)

# Vis de første 2 chunks med sine embeddings
for i, chunk in enumerate(chunks[:2]):
    print(f"\n{'🔹' * 35}")
    print(f"CHUNK {i+1}")
    print(f"{'🔹' * 35}")
    
    # Vis chunk-teksten
    print(f"\n📝 TEKST ({len(chunk['text'])} tegn):")
    print("-" * 50)
    preview = chunk['text'][:500] + "..." if len(chunk['text']) > 500 else chunk['text']
    print(preview)
    
    # Hent og vis embedding
    embedding = get_embedding(chunk['text'])
    
    print(f"\n🔢 EMBEDDING VEKTOR:")
    print("-" * 50)
    print(f"   Dimensjoner: {len(embedding)}")
    print(f"   Første 5:  {embedding[:5]}")
    print(f"   Siste 5:   {embedding[-5:]}")
    print(f"   Min/Max:   {min(embedding):.4f} / {max(embedding):.4f}")
    
    # Lagre embedding for senere sammenligning
    chunk['embedding'] = embedding

print(f"\n{'=' * 70}")
print("💡 Legg merke til at hver chunk får sin egen unike embedding-vektor!")
print("   Lignende tekster vil ha lignende vektorer (høy cosine similarity / lik semantisk verdi)")
print("=" * 70)

📄 CHUNK + EMBEDDING INSPEKSJON

🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
CHUNK 1
🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹

📝 TEKST (62 tegn):
--------------------------------------------------
--- Side 1 ---
Delårsrapport          
per tredje kvartal 2025

🔢 EMBEDDING VEKTOR:
--------------------------------------------------
   Dimensjoner: 3072
   Første 5:  [0.022716378793120384, -0.003645680844783783, -0.00963957142084837, 0.0017824274254962802, -0.0020759536419063807]
   Siste 5:   [0.011953749693930149, -0.0004440115881152451, -0.009648079983890057, 0.0010884931543841958, -0.00748704606667161]
   Min/Max:   -0.0849 / 0.1076

🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹
CHUNK 2
🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹🔹

📝 TEKST (789 tegn):
--------------------------------------------------
--- Side 2 ---
Introduksjon
Hafslund er et fornybarkonsern bestående av tre forretningsområder: 
Kraftproduksjon, med Norges nest største vannkraftvirksomhet, 
Fjernvarme, som er Norges største fjernvarmeleverandør og V

### Steg 4: Indekser alle chunks i vektordatabasen

Nå indekserer vi alle chunks i ChromaDB. For hver chunk:
1. Genererer vi en embedding-vektor
2. Lagrer tekst + vektor + metadata i databasen

In [9]:
# Indekser alle chunks i ChromaDB
print("🚀 Indekserer alle chunks...")
print("-" * 50)

for i, chunk in enumerate(chunks):
    embedding = get_embedding(chunk["text"])
    
    collection.add(
        ids=[chunk["id"]],
        embeddings=[embedding],
        documents=[chunk["text"]],
        metadatas=[chunk["metadata"]]
    )
    
    # Vis fremdrift
    if (i + 1) % 10 == 0 or i == len(chunks) - 1:
        print(f"   ✅ Indeksert {i + 1}/{len(chunks)} chunks")

print(f"\n🎉 Ferdig! {collection.count()} chunks er nå søkbare!")

🚀 Indekserer alle chunks...
--------------------------------------------------
   ✅ Indeksert 10/69 chunks
   ✅ Indeksert 10/69 chunks
   ✅ Indeksert 20/69 chunks
   ✅ Indeksert 20/69 chunks
   ✅ Indeksert 30/69 chunks
   ✅ Indeksert 30/69 chunks
   ✅ Indeksert 40/69 chunks
   ✅ Indeksert 40/69 chunks
   ✅ Indeksert 50/69 chunks
   ✅ Indeksert 50/69 chunks
   ✅ Indeksert 60/69 chunks
   ✅ Indeksert 60/69 chunks
   ✅ Indeksert 69/69 chunks

🎉 Ferdig! 69 chunks er nå søkbare!
   ✅ Indeksert 69/69 chunks

🎉 Ferdig! 69 chunks er nå søkbare!


### Steg 5: Søk i dokumentet

Nå kan vi søke etter relevante chunks basert på et spørsmål. Søket fungerer ved at:
1. Spørsmålet konverteres til en embedding
2. Vi finner chunks med lignende embeddings (cosine similarity)
3. De mest relevante chunks returneres

In [10]:
def search_documents(query: str, n_results: int = 3) -> list[dict]:
    """Søk etter relevante dokumenter basert på semantisk likhet."""

    # Hent embedding for spørsmålet
    query_embedding = get_embedding(query)
    
    # Gør en semantisk søk i collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    
    return [
        {
            "text": doc,
            "metadata": meta,
            "distance": dist
        }
        for doc, meta, dist in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["distances"][0]
        )
    ]

# Test søk
spørsmål = "Hvordan har vannkraftproduksjonen vært?"
resultater = search_documents(spørsmål, n_results=5)

print(f"🔍 Spørsmål: {spørsmål}\n")
print("📚 Relevante dokumenter:")
for i, r in enumerate(resultater, 1):
    print(f"\n{i}. [Score: {1-r['distance']:.2f}]")
    print(f"   {r['text'][:200]}...")

🔍 Spørsmål: Hvordan har vannkraftproduksjonen vært?

📚 Relevante dokumenter:

1. [Score: 0.63]
   som verdsettes til markedsverdi i resultatet. 
Kraftproduksjonen på 12,8 TWh (14,1 TWh) er ned 9 prosent fra fjoråret. 
Dette bidro isolert sett til 839 millioner kroner i redusert driftsresultat mot ...

2. [Score: 0.61]
   til markedsverdi i resultatet, var det underliggende driftsresultatet 7 141 
millioner kroner (6 554 millioner kroner), som tilsvarer en økning på 9 
prosent fra fjoråret. 
Oppnådd kraftpris på 68 øre...

3. [Score: 0.60]
   produksjonsområder og må ses i sammenheng med løpende 
produksjonsoptimalisering, inntjening i balansemarkedet og 
sikringsaktivitet.
• Kraftproduksjonen på 12,9 TWh per tredje kvartal 2025 er ned 9 
...

4. [Score: 0.58]
   --- Side 6 ---
Forretningsområdene 
Kraftproduksjon
Mill. kroner
HiÅ 2025
HiÅ 2024
2024
Driftsinntekter
 
9 427  
8 410  
11 751 
EBITDA
 
7 885  
7 021  
9 707 
Driftsresultat (EBIT)
 
7 210  
6 531 ...

5. [Score: 0.53]
   ve

### Steg 6: RAG - Kombiner søk med LLM

Nå setter vi sammen hele RAG-pipelinen: søk + generering.

In [11]:
def rag_query(question: str) -> dict:
    """Svar på spørsmål med RAG."""
    
    # 1. Hent relevante dokumenter
    relevant_docs = search_documents(question, n_results=3)
    
    # 2. Bygg kontekst
    context = "\n\n".join([doc["text"] for doc in relevant_docs])
    
    # 3. Lag prompt med kontekst
    system_prompt = """Du er en hjelpsom assistent som svarer på spørsmål basert på den gitte konteksten.
    
Regler:
- Svar KUN basert på informasjonen i konteksten
- Hvis svaret ikke finnes i konteksten, si "Jeg finner ikke informasjon om dette i dokumentene."
- Vær konsis og presis
- Svar på norsk"""
    
    user_prompt = f"""Kontekst:
{context}

Spørsmål: {question}"""
    
    # 4. Kall Azure OpenAI LLM
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    
    # 5. Returner svar + chunk-indekser som kilder
    return {
        "answer": response.choices[0].message.content,
        "kilder": [doc["metadata"]["chunk_index"] for doc in relevant_docs]
    }

print("✅ RAG-funksjon klar!")

✅ RAG-funksjon klar!


### 🎯 Test RAG!

In [12]:
# Test med spørsmål om delårsrapporten
spørsmål_liste = [
    "Fortell meg om Karbonfangstprosjektet på Klemetsrud.",
    "Hvordan har vannkraftproduksjonen vært?",
    "Hva er de viktigste finansielle nøkkeltallene?",
]

for spørsmål in spørsmål_liste:
    print(f"\n{'='*70}")
    print(f"❓ {spørsmål}")
    print(f"{'='*70}")
    
    resultat = rag_query(spørsmål)
    
    print(f"\n💬 {resultat['answer']}")
    print(f"\n📄 Kilder: chunk {resultat['kilder']}")


❓ Fortell meg om Karbonfangstprosjektet på Klemetsrud.

💬 Karbonfangstprosjektet på Klemetsrud er et pågående arbeid med å bygge det som vil bli et av verdens første karbonfangstanlegg for avfallsforbrenning. I september ble AF Gruppen innstilt som entreprenør for bygging av nytt teknisk bygg og fundamenter. Etter endelig investeringsbeslutning i januar 2025 er forberedende arbeider i gang før selve byggingen starter.

Anlegget forventes å være i drift i tredje kvartal 2029 og vil redusere de fossile CO₂-utslippene i Oslo med rundt 20 prosent. Hafslund Celsio har inngått avtaler om salg av sertifikater for permanent karbonfjerning, blant annet med Microsoft for 1,1 millioner tonn over ti år og med Frontier for totalt 100 000 tonn i 2029 og 2030. Avtalene sikrer betydelige årlige inntekter de første ti årene etter idriftsettelse og bidrar til lønnsomheten i prosjektet.

📄 Kilder: chunk [12, 27, 13]

❓ Hvordan har vannkraftproduksjonen vært?

💬 Karbonfangstprosjektet på Klemetsrud er et

## 📝 RAG Oppsummering

**Fordeler med RAG:**
- ✅ Oppdatert informasjon uten re-trening
- ✅ Kontroll over datakilder
- ✅ Reduserer hallusinasjoner
- ✅ Sporbarhet - vet hvilke dokumenter som brukes

**Utfordringer:**
- ⚠️ Kvalitet avhenger av chunking-strategi
- ⚠️ Krever god embedding-modell
- ⚠️ Kan være tregt med store datamengder

---

# MCP - Model Context Protocol

## Hva er MCP?

**MCP** (Model Context Protocol) er en åpen standard fra Anthropic som lar AI-modeller koble seg til **eksterne verktøy og datakilder** på en standardisert måte.

### Hvorfor MCP?

Før MCP måtte hver AI-integrasjon bygges fra bunnen av. Med MCP får vi:

- 🔌 **Plug-and-play** - Ett grensesnitt for alle verktøy
- 🔄 **Gjenbrukbare komponenter** - Servere kan deles på tvers av prosjekter  
- 🛡️ **Sikkerhet** - Kontrollert tilgang til ressurser
- 🌐 **Økosystem** - Voksende bibliotek av ferdige MCP-servere

### MCP Arkitektur

```
┌─────────────────────────────────────────────────────────────────┐
│                     MCP Arkitektur                              │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│   ┌─────────────────────────────────────────────────────────┐   │
│   │                    MCP HOST                             │   │
│   │              (Claude, VS Code, etc.)                    │   │
│   │                                                         │   │
│   │   ┌──────────────────────────────────────────────────┐  │   │
│   │   │                 MCP CLIENT                       │  │   │
│   │   │         (håndterer kommunikasjon)                │  │   │
│   │   └──────────────────────────────────────────────────┘  │   │
│   └─────────────────────────┬───────────────────────────────┘   │
│                             │                                   │
│                             │ JSON-RPC over stdio/SSE           │
│                             │                                   │
│   ┌─────────────────────────┴───────────────────────────────┐   │
│   │                                                         │   │
│   │  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │   │
│   │  │ MCP SERVER   │  │ MCP SERVER   │  │ MCP SERVER   │   │   │
│   │  │              │  │              │  │              │   │   │
│   │  │ 📁 Filsystem │  │ 🗄️ Database  │  │ 🌐 API       │   │   │
│   │  │              │  │              │  │              │   │   │
│   │  └──────────────┘  └──────────────┘  └──────────────┘   │   │
│   │                                                         │   │
│   └─────────────────────────────────────────────────────────┘   │
│                                                                 │
│   MCP Servere eksponerer:                                       │
│   • 🔧 Tools    - Funksjoner AI kan kalle                       │
│   • 📚 Resources - Data AI kan lese                             │
│   • 💬 Prompts  - Ferdiglagde prompt-templates                  │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

## 🛠️ MCP i praksis

Vi bruker:
- **FastMCP** - For å lage MCP-servere
- **langchain-mcp-adapters** - For å koble til servere som klient

### Våre verktøy

| Verktøy | Beskrivelse |
|---------|-------------|
| `get_current_time` | Henter nåværende tid i Norge |
| `get_weather` | Værdata fra Yr/Met.no (krever lat/lon) |
| `get_hydro_stations` | Hydrostasjoner fra NVE HydAPI |
| `get_water_level` | Vannstand for en spesifikk stasjon |

### Ferdige MCP-servere

| Server | Beskrivelse |
|--------|-------------|
| [yr-mcp-server](https://github.com/AnyContext-ai/yr-mcp-server) | Værdata fra yr.no |
| [filesystem](https://github.com/modelcontextprotocol/servers) | Les/skriv filer |
| [fetch](https://github.com/modelcontextprotocol/servers) | Hent data fra URL-er |

> 💡 Se [MCP Servers Directory](https://github.com/modelcontextprotocol/servers) for full liste!

In [13]:
from fastmcp import FastMCP
from datetime import datetime
import httpx
import os

mcp = FastMCP("Workshop Server")

# Hjelpefunksjon for NVE API-kall
def _nve_request(endpoint: str, params: dict = None) -> dict:
    """Utfør request mot NVE HydAPI."""
    api_key = os.getenv("HYDRO_API_KEY")
    if not api_key:
        raise ValueError("HYDRO_API_KEY ikke satt")
    
    response = httpx.get(
        f"https://hydapi.nve.no/api/v1/{endpoint}",
        headers={"X-API-Key": api_key},
        params=params,
        timeout=30
    )
    response.raise_for_status()
    return response.json()

@mcp.tool()
def get_current_time() -> dict:
    """Hent nåværende dato og klokkeslett. Returnerer {time, date}."""
    now = datetime.now()
    return {"time": now.strftime("%H:%M:%S"), "date": now.strftime("%Y-%m-%d")}

@mcp.tool()
def get_weather(lat: float, lon: float) -> dict:
    """Hent værdata for koordinater. Eksempel: lat=59.91, lon=10.75 (Oslo). Returnerer {temperature, wind}."""
    response = httpx.get(
        f"https://api.met.no/weatherapi/locationforecast/2.0/compact?lat={lat}&lon={lon}",
        headers={"User-Agent": "GenAI-Workshop/1.0"},
        timeout=10
    )
    response.raise_for_status()
    details = response.json()["properties"]["timeseries"][0]["data"]["instant"]["details"]
    return {
        "temperature": f"{details['air_temperature']}°C",
        "wind": f"{details['wind_speed']} m/s"
    }

@mcp.tool()
def get_hydro_stations(county: str = None, limit: int = 3) -> list[dict]:
    """Søk etter hydrostasjoner i Norge. Bruk county for fylkesfilter (f.eks. 'Oslo', 'Vestland'). Returnerer [{id, navn}]."""
    data = _nve_request("Stations", {"Active": "1"})
    stations = data.get("data", data)
    
    if county:
        stations = [s for s in stations if county.lower() in s.get("countyName", "").lower()]
    
    return [{"id": s["stationId"], "navn": s.get("stationName")} for s in stations[:limit]]

@mcp.tool()
def get_water_level(station_id: str) -> dict:
    """Hent vannstand for en stasjon. Bruk station_id fra get_hydro_stations. Returnerer {vannstand_m, tid}."""
    data = _nve_request("Observations", {"StationId": station_id, "Parameter": "1000", "ResolutionTime": "0"})
    
    if not data.get("data") or not data["data"][0].get("observations"):
        return {"station_id": station_id, "error": "Ingen data"}
    
    obs = data["data"][0]["observations"][-1]
    return {"station_id": station_id, "vannstand_m": obs["value"], "tid": obs["time"]}

print("✅ MCP Server klar med 4 verktøy")

✅ MCP Server klar med 4 verktøy


### Start serveren

Kjør serveren i bakgrunnen på port 8005:

In [14]:
import threading, time
import warnings

# Ignorer deprecation warnings fra websockets (uvicorn-avhengighet)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="websockets")

PORT = 8005

# Start serveren i bakgrunnen (eneste måte i notebooks)
def _run():
    import uvicorn
    uvicorn.run(mcp.http_app(), host="127.0.0.1", port=PORT, log_level="error")

threading.Thread(target=_run, daemon=True).start()
time.sleep(1)

print(f"✅ MCP Server kjører på http://127.0.0.1:{PORT}/mcp")


/Users/vegard.johnsen/Library/CloudStorage/OneDrive-Hafslund/Skrivebord/genai.workshop/.venv/lib/python3.13/site-packages/uvicorn/protocols/websockets/websockets_impl.py:17: DeprecationWarning: websockets.server.WebSocketServerProtocol is deprecated
  from websockets.server import WebSocketServerProtocol


✅ MCP Server kjører på http://127.0.0.1:8005/mcp


### Koble til med MCP Client

Bruk `MultiServerMCPClient` fra langchain-mcp-adapters:

In [15]:
import warnings
warnings.filterwarnings("ignore", message="Use `streamable_http_client` instead")

from langchain_mcp_adapters.client import MultiServerMCPClient

# Opprett MCP-klient
mcp_client = MultiServerMCPClient({
    "demo": {
        "url": f"http://127.0.0.1:{PORT}/mcp",
        "transport": "http"
    }
})

# Hent alle tilgjengelige verktøy
tools = await mcp_client.get_tools()

print("🔧 Tilgjengelige MCP-verktøy:")
for tool in tools:
    print(f"  • {tool.name}: {tool.description}")

🔧 Tilgjengelige MCP-verktøy:
  • get_current_time: Hent nåværende dato og klokkeslett. Returnerer {time, date}.
  • get_weather: Hent værdata for koordinater. Eksempel: lat=59.91, lon=10.75 (Oslo). Returnerer {temperature, wind}.
  • get_hydro_stations: Søk etter hydrostasjoner i Norge. Bruk county for fylkesfilter (f.eks. 'Oslo', 'Vestland'). Returnerer [{id, navn}].
  • get_water_level: Hent vannstand for en stasjon. Bruk station_id fra get_hydro_stations. Returnerer {vannstand_m, tid}.


### 🤖 Gi språkmodellen tilgang på verktøyene som er satt opp ovenfor

Her lar vi språkmodellen selv bestemme hvilket verktøy den skal bruke, og hvilke argumenter (som lat/lon) den skal sende:

In [18]:
from langchain_openai import AzureChatOpenAI
from langchain.agents import create_agent

# Sett opp LangChain LLM
llm = AzureChatOpenAI(
    azure_deployment=CHAT_MODEL,
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-15-preview"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

# Lag agent med MCP-verktøy
agent = create_agent(llm, tools)

print("✅ Agent klar!")

✅ Agent klar!


In [23]:
# Test agenten - den velger selv hvilke verktøy den trenger
result = await agent.ainvoke(
    {"messages": [{"role": "user", "content": "Gi meg vannstand for kraftverkene i Oslo"}]},
    debug=True  # Vis hva agenten gjør
)

print(f"\n💬 {result['messages'][-1].content}")

[values] {'messages': [HumanMessage(content='Gi meg vannstand for kraftverkene i Oslo', additional_kwargs={}, response_metadata={}, id='a9e91e0c-c8ce-4b96-b018-817b163e65b3')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 307, 'total_tokens': 403, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2-chat-2025-12-11', 'system_fingerprint': None, 'id': 'chatcmpl-CxV0eSBUdxiU5lIItJfbJxlEP9uJX', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'saf

## 📝 MCP Oppsummering

**Fordeler med MCP:**
- ✅ Standardisert protokoll for AI-verktøy
- ✅ Separasjon mellom AI og verktøy
- ✅ Sikker og kontrollert tilgang
- ✅ Voksende økosystem av ferdige servere

**Populære MCP-servere:**
- 📁 Filesystem - Les/skriv filer
- 🗄️ Database - PostgreSQL, SQLite
- 🌐 Web - Fetch, Puppeteer
- 🔧 DevTools - Git, GitHub, Slack

---

### 🔌 Bonus: Koble til flere MCP-servere

Du kan koble til **flere MCP-servere samtidig** med `MultiServerMCPClient`. Her er et eksempel med yr-mcp-server:

```bash
# Installer yr-mcp-server
git clone https://github.com/AnyContext-ai/yr-mcp-server
cd yr-mcp-server
pip install -r requirements.txt
python yr.py  # Starter på port 8000
```

```python
# Koble til flere servere samtidig
mcp_client = MultiServerMCPClient({
    "workshop": {
        "url": "http://127.0.0.1:8005/mcp",  # Vår server
        "transport": "http"
    },
    "yr": {
        "url": "http://127.0.0.1:8000/mcp",  # yr-mcp-server
        "transport": "http"
    }
})

# Alle verktøy fra begge servere blir tilgjengelige for agenten!
tools = await mcp_client.get_tools()
```

```

> 💡 Agenten kan nå bruke verktøy fra **alle** tilkoblede servere automatisk!

# 🎓 Oppsummering

## Hva vi har lært:

| Teknologi | Hovedkonsept | Bruksområde |
|-----------|--------------|-------------|
| **RAG** | Hent kontekst → Generer svar | Snakk med egne dokumenter |
| **MCP** | Standardisert verktøy-protokoll | Koble AI til API-er og systemer |

## Neste steg:

1. **Eksperimenter med RAG** på egne dokumenter
2. **Bygg en MCP-server** for interne systemer
3. **Kombiner RAG + MCP** for kraftige AI-assistenter

## Ressurser:

- 📖 [LangChain RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/)
- 📖 [MCP Dokumentasjon](https://modelcontextprotocol.io/)
- 📖 [FastMCP GitHub](https://github.com/jlowin/fastmcp)
- 📖 [ChromaDB Docs](https://docs.trychroma.com/)

---
